In [ ]:
import sys

sys.path.append('..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from src.flow_matching.helpers import choose_device
from src.flow_matching.loader import (
    load_model_info,
    load_trained_model,
    prob_path_from_config,
)
from src.flow_matching.neural_posterior import NeuralPosterior
from src.flow_matching.simulator import Model as BatchedModel

device = choose_device()

# Loading in the FM model

In [ ]:
# fill in desired job_id or directory name 
job_id = "15055686" 

# loading the model (found via job_id, or path)
vector_field, classifier, config = load_trained_model(job_id, device, ema=False)

# load probability path used in training
path = prob_path_from_config(config)

# load useful training parameters
model_info = load_model_info(config, path=path, device=device)

In [ ]:
inf_params  = model_info['inf_params']
N           = model_info['N']
vector_dim  = model_info['vector_dim']
burstparams = model_info['burstparams']
mean        = model_info['mean']
std         = model_info['std']

transdimensional = model_info['transdimensional']
YBKG        = config['path']['p_data']['init_params']['model_params']['ybkg']

In [ ]:
# making sure model is valid for magnetar data
if path.p_data.noise != 'poisson':
    raise ValueError('Model not trained with poisson noise.')

# generate FM posterior conditioned on **observed** counts

## Read observational magnetar burst data sample

In [ ]:
filepath = '../' 'observational_data/magnetar_bursts/090122173_+241.347_all_data.dat' # for 2015 comparison
# filepath = '../' 'observational_data/magnetar_bursts/090122173_-000.022_all_data.dat' 
# filepath = '../' 'observational_data/magnetar_bursts/090122173_+139.209_all_data.dat' 
# filepath = '../' 'observational_data/magnetar_bursts/090122173_+037.024_all_data.dat'
# filepath = '../' 'observational_data/magnetar_bursts/090122173_+147.522_all_data.dat'
# filepath = '../' 'observational_data/magnetar_bursts/090122173_+271.607_all_data.dat'
# filepath = '../' 'observational_data/magnetar_bursts/090122173_+290.888_all_data.dat'
# filepath = '../' 'observational_data/magnetar_bursts/090122173_+293.748_all_data.dat'

burst_name = filepath.split('/')[-1]

def read_magnetar_data(filepath):
    df = pd.read_csv(filepath, header=None, delimiter=' ', usecols=[1])
    observed_counts = torch.tensor(df[1], device=device) 
    return observed_counts

observed_counts = read_magnetar_data(filepath)
plt.plot(observed_counts.cpu())
plt.title(f'{burst_name}')
observed_counts.shape

In [ ]:
def downsample(profile, factor):
    """
    Reduce time resolution by merging bins.
    """
    for i in range(factor):
        bins_1 = profile[::2]
        bins_2 = profile[1::2]
        if len(bins_1) != len(bins_2):
            bins_1_new = torch.zeros_like(bins_2)
            bins_1_new = bins_1[:-1]
            bins_1 = bins_1_new
        profile = bins_1 + bins_2
    return profile 

def pad_with_noise(profile, noise_level, noise:str, new_length=1000):
    """Symmetrically pad burst with poisson or gaussian noise.
    """
    assert len(profile) < new_length, 'profile exceeds desired length, padding not possible.'
    assert noise in ['poisson', 'gaussian'], 'noise must be either gaussian or poisson.'

    noise_padding_size = new_length - len(profile)
    padding_left = noise_padding_size // 2 + noise_padding_size % 2
    padding_right = noise_padding_size // 2
    
    if noise == "poisson":
        noise_left = torch.poisson(torch.ones(padding_left, device=device) * noise_level)
        noise_right = torch.poisson(torch.ones(padding_right, device=device) * noise_level)

    elif noise == "gaussian": 
        noise_left = torch.randn(int(padding_left), device=device) * noise_level
        noise_right = torch.randn(int(padding_right), device=device) * noise_level

    prepped_counts = torch.zeros(new_length)
    prepped_counts[:padding_left] = noise_left
    prepped_counts[new_length - padding_right:] = noise_right
    prepped_counts[padding_left:new_length - padding_right] = profile 
    return prepped_counts, padding_left, new_length - padding_right

In [ ]:
# prep magnetar burst

# reduce time resolution dt 5e-4 -> 1e-3
downsampled = downsample(profile=observed_counts, factor=1)
prepped_counts, profile_start, profile_end = pad_with_noise(downsampled, noise_level=3, noise='poisson', new_length=1000)

observed_counts = prepped_counts 

In [ ]:
plt.plot(observed_counts)
plt.title('prepped burst')
plt.show()

In [ ]:
learned_posterior = NeuralPosterior(
    vector_field,
    classifier,
    model_info,
    path,
    device=device
)

In [ ]:
num_samples = 1000
samples_per_batch = 100

Ns, theta_samples = learned_posterior.sample(num_samples, observed_counts, samples_per_batch, time_steps=200)
p_N = learned_posterior.p_N
Ns = Ns.to('cpu')
theta_samples

# Classifier p(N)

In [ ]:
plt.bar(range(1, len(p_N.cpu().detach().numpy())+1), height=p_N.cpu().detach().numpy())
plt.xlabel("$N_{pred}$")
plt.title('p(N|y)')

print("Most likely estimate from classifier: ", range(1, len(p_N.cpu().detach().numpy())+1)[torch.where(p_N == torch.max(p_N))[0]])

## posterior samples

In [ ]:
# posterior samples that include all sampled component numbers
num_samples = 1000
posterior_param_samples = path.p_data.prior.samples_as_dict(theta_samples)

posterior_curve_samples = BatchedModel(
    device=device, 
    time=torch.linspace(0, 1, 1000), 
    burstparams=posterior_param_samples, ybkg=YBKG, ncomp=Ns
        ).get_flux()

import matplotlib.gridspec as gridspec

# remove noise for plotting
observed_counts_no_pad = observed_counts[profile_start:profile_end]
posterior_curve_samples_no_pad = posterior_curve_samples[:, profile_start:profile_end]

fig, ax1 = plt.subplots(1, 1)
fig.set_figheight(5.8)
fig.set_figwidth(6)
# fig = plt.figure(figsize=(11, 4))
# gs = gridspec.GridSpec(1, 2, width_ratios=[1.5, 1])  # first subplot wider

# ax1 = plt.subplot(gs[0])
# ax2 = plt.subplot(gs[1])
time_res = 1e-3
ax1.plot(np.linspace(0, observed_counts_no_pad.shape[0] * time_res, observed_counts_no_pad.shape[0]), observed_counts_no_pad.flatten().cpu(),'k-', alpha=0.6,  label='observed flux')
sample_colors=['pink', 'yellow', 'orange', 'purple', 'green', 'cyan', 'red', 'grey', 'magenta', 'blue']

# plot mean curve
# mean_profile = torch.median(posterior_curve_samples[(Ns==12).flatten()], dim=0)[0].cpu()
mean_profile = torch.mean(posterior_curve_samples_no_pad[0:num_samples], dim=0).cpu()
std_profile  = torch.std(posterior_curve_samples_no_pad[0:num_samples], dim=0).cpu()

# 95% interval of data
q1 = torch.quantile(posterior_curve_samples_no_pad[0:num_samples], 0.025, dim=0).cpu()
q3 = torch.quantile(posterior_curve_samples_no_pad[0:num_samples], 0.975, dim=0).cpu()
iqr = q3 - q1

time_ = np.linspace(0, posterior_curve_samples_no_pad.shape[1] * time_res, posterior_curve_samples_no_pad.shape[1])
ax1.plot(time_, mean_profile, color='white', label='mean profile', linewidth=1)
ax1.fill_between(time_, q1, q3, color='red', alpha=0.7, label='std profile')

# styling of left plot 
obs_id = filepath.split('/')[-1].split('_')[0]
start_time = filepath.split('/')[-1].split('_')[1]
name_id = obs_id +'_'+start_time
# ax1.set_title(name_id, fontsize=16)
ax1.grid(linestyle='dotted')
ax1.set_ylabel('count rate (10$^3$ counts s$^{-1}$)', fontsize=14)
ax1.set_xlabel('t (s)', fontsize=14)
ax1.set_xlim(0.0,posterior_curve_samples_no_pad.shape[1] * time_res)
ax1.tick_params(axis='both', labelsize=14)

# plt.subplot(122)

# predicted components probabilities
# ax2.bar(
#     range(1, len(p_N.cpu().detach().numpy())+1), 
#     height=p_N.cpu().detach().numpy(), 
#     # color=sample_colors,
#     color='lightsteelblue', 
#     edgecolor='black',    # color of the bar edge
#     linewidth=1,
#     label=range(1, 21)
#     )
# # ax2.legend()
# ax2.set_xlabel("$N$", fontsize=14)
# ax2.set_ylabel('$p_\phi(N\mid y)$', fontsize=14, labelpad=15)
# ax2.tick_params(axis='both', labelsize=14)
# ax2.set_xlim(right=20)
# plt.xlim(9, 21)
# # plt.grid(axis='y', linestyle='dotted', zorder=-100)

# # integer tick labels for component number
# import matplotlib.ticker as mticker
# ax2.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig(f'{name_id}.png', dpi=300)
plt.show()


In [ ]:
# similar layout to 2015 paper for comparison
plt.figure(figsize=(6,5.8))
plt.plot(np.linspace(0, 1, observed_counts.shape[0]), observed_counts.flatten().cpu(),'k-', alpha=0.7,  label='observed flux')
sample_colors=['pink', 'yellow', 'orange', 'purple', 'green', 'cyan', 'red', 'grey', 'magenta', 'blue']

# only 10 samples
for i in range(10):
    random_idx = i#torch.randint(0, 10, size=(1,)).item()
    plt.plot(
        np.linspace(0, 1, observed_counts.shape[0]), 
        posterior_curve_samples[random_idx].cpu(), 
        label='posterior samples' if i==0 else ''
        )
# plt.ylim(top=8.5)
# ax1.legend(fontsize=12, loc='upper right')
plt.grid(linestyle='dotted')
plt.ylabel('counts s$^{-1}$', fontsize=14)
plt.xlabel('t', fontsize=14)
plt.xlim(0.1,0.9)
plt.ylim(0)
# ax1.set_ylim(top=11)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.show()

plt.figure(figsize=(6,5.8))
plt.grid(linestyle='dotted', axis='y')

plt.bar(
    range(1, len(p_N.cpu().detach().numpy())+1), 
    height=p_N.cpu().detach().numpy(), 
    # color=sample_colors,
    color='lightsteelblue', 
    edgecolor='black',    # color of the bar edge
    linewidth=1,
    width=0.4,
    zorder=1
    # label=range(1, 21)
    )
# plt.grid(zorder=-1)
plt.ylim(0, 0.205)
plt.xlabel("$N$", fontsize=14)
plt.ylabel('$p_\phi(N\mid y)$', fontsize=14, labelpad=15)
plt.xticks(fontsize=14)
plt.yticks(ticks=[0, 0.04, 0.08, 0.12,0.16,0.2], fontsize=14)
plt.xlim(9, 21)